# 免费GPU跑真实蛋白-配体复合物MD

这个notebook把这次会话里在Mac CPU上真实跑过的MD代码(`core/structures.py::run_protein_equilibration_md()`、`core/md_stability.py::run_protein_ligand_complex_md()`)搬到Colab的免费GPU上跑，规模从demo级(10ps)提升到更接近真实项目需要的量级。

**完全自包含**：不需要上传这个仓库，所有结构数据都会在notebook里重新从RCSB真实下载+重建，用的是这次会话里已经验证过的同一套方法(真实6LUD晶体结构 + 真实奥希替尼晶体坐标，用`AllChem.AssignBondOrdersFromTemplate`转移正确键级)。

**用法**：
1. 打开 colab.research.google.com，上传这个文件(文件 → 上传笔记本)
2. 菜单栏「代码执行程序」→「更改运行时类型」→ 硬件加速器选GPU(T4)——这个notebook的元数据已经标了GPU，大概率会自动选上，但建议手动确认一下
3. **第一个代码cell(装condacolab)必须单独运行，不要用"全部运行"**——运行完它会自动重启整个运行时，中途会弹出一个"会话已崩溃"/"Your session crashed"的提示，**这是正常的，是condacolab设计上就会这样**，不是出错了。重启之后，从下一个cell接着往下跑。

每一步都会打印真实测出来的数字(原子数/能量/耗时)，不是预先写好的假输出。

## 第1步：装condacolab(这一个cell要单独运行，运行完会自动重启，看到崩溃提示不要慌)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()
# 运行到这里会自动重启运行时，下面的cell要等重启完成之后再运行

## 第2步：重启之后，从这里继续——装这次会话真实用过的那一套依赖

和本地跑这次会话一样的组合：`ambertools`(真实antechamber二进制，给配体做GAFF2参数化)+ `openff-toolkit-base` + `openmm` + `openmmforcefields` + `pdbfixer` + `rdkit`。用`mamba`不用`conda`classic——后者解这组依赖可能卡在"Solving environment"几十分钟，这次会话本地踩过这个坑。

**真实查证过的坑，装`openff-toolkit`(完整版)而不是`openff-toolkit-base`会踩到**：完整版`openff-toolkit`会带出可选的ML电荷后端`openff-nagl`，`openff-nagl`又带出`pytorch`全家桶(pytorch-lightning/torchmetrics)，而`torchmetrics`会在导入时尝试给`torchvision::nms`这个算子注册一个"假"实现——但环境里从头到尾都没真正装`torchvision`，这个算子从没被真正注册过，于是报`RuntimeError: operator torchvision::nms does not exist`。本仓库用gasteiger电荷，完全不需要`openff-nagl`这个ML后端，查过OpenFF官方文档：`openff-toolkit-base`只装核心硬依赖，不会带出这整条可选的ML链路(我们另外显式装的`rdkit`/`ambertools`已经覆盖了`openff-toolkit`完整版本来想顺带捆绑的那部分东西)。这是这次踩这个坑的第4次尝试，前三次(钉Python版本/卸载torch/补装torchvision)都是在尝试"修好"这整条ML链路，这次是直接换成不触发这条链路的包，方向不一样。

In [ ]:
import condacolab
condacolab.check()  # 确认重启后环境是好的，如果这一步报错，重新执行一次第1步的cell

!mamba install -y -q -c conda-forge ambertools openff-toolkit-base openmm openmmforcefields pdbfixer rdkit

In [ ]:
# 验证安装 + 检查有没有真的分到GPU(CUDA平台)——这是全部后续步骤的前提
import openmm
from openmm import Platform

print("OpenMM version:", openmm.__version__)
platform_names = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
print("可用平台:", platform_names)

if "CUDA" in platform_names:
    print("\n✅ 真的分到GPU了，后面会用CUDA平台跑，比本地Mac的CPU快得多")
else:
    print("\n⚠️ 没有看到CUDA平台——去菜单栏「代码执行程序」→「更改运行时类型」确认选的是GPU，"
          "然后重启运行时重新跑一遍。如果Colab这次没分到GPU(免费层偶尔会遇到)，"
          "后面的代码会自动退回CPU/OpenCL，能跑但不会有GPU加速。")

import subprocess
r = subprocess.run(["which", "antechamber"], capture_output=True, text=True)
print("\nantechamber路径:", r.stdout.strip() or "(没找到，GAFF参数化会失败，检查上面mamba install是否真的成功)")

## 第3步：真实下载6LUD晶体结构 + 清洗出受体(和这次会话本地跑的是同一个真实PDB)

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://files.rcsb.org/download/6LUD.pdb", "6lud_raw.pdb")
print("真实下载完成:", __import__("os").path.getsize("6lud_raw.pdb"), "bytes")


def clean_chain(raw_pdb_path, chain, out_path, keep_hetatm=False):
    """和core/structures.py::clean_chain()完全一样的逻辑，这里内嵌一份避免依赖整个仓库。"""
    n_atom, n_hetatm = 0, 0
    with open(raw_pdb_path) as f_in, open(out_path, "w") as f_out:
        for line in f_in:
            record_type = line[:6].strip()
            if len(line) < 22:
                continue
            chain_col = line[21]
            if record_type == "ATOM" and chain_col == chain:
                f_out.write(line)
                n_atom += 1
            elif keep_hetatm and record_type == "HETATM" and chain_col == chain:
                f_out.write(line)
                n_hetatm += 1
    return {"out_path": out_path, "n_atom_lines": n_atom, "n_hetatm_lines": n_hetatm}


result = clean_chain("6lud_raw.pdb", chain="A", out_path="6lud_receptor_only.pdb", keep_hetatm=False)
print(result)

## 第4步：从真实晶体坐标重建奥希替尼(真实键级，不是随便embed的构象)

和这次会话本地做法完全一样：从6LUD真实HETATM坐标解析出来，用已验证过的真实SMILES(InChIKey核对过的)做模板转移键级信息。

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors

pdb_lines = []
with open("6lud_raw.pdb") as f:
    for line in f:
        if line.startswith("HETATM") and line[17:20].strip() == "YY3" and line[21] == "A":
            pdb_lines.append(line)
print("真实YY3(奥希替尼)HETATM行数:", len(pdb_lines))

pdb_block = "HEADER\n" + "".join(pdb_lines) + "END\n"
raw_mol = Chem.MolFromPDBBlock(pdb_block, sanitize=False, removeHs=False)

# 这个SMILES在tests/test_standardize.py里验证过InChIKey和真实公开数据库一致
template_smiles = "COc1cc(N(C)CCN(C)C)c(NC(=O)C=C)cc1Nc1nccc(-c2cn(C)c3ccccc23)n1"
template = Chem.MolFromSmiles(template_smiles)
real_mol = AllChem.AssignBondOrdersFromTemplate(template, raw_mol)
print("真实坐标+正确键级，分子式:", rdMolDescriptors.CalcMolFormula(real_mol), "(真实奥希替尼应该是C28H33N7O2)")

real_mol_h = Chem.AddHs(real_mol, addCoords=True)
print("补氢后原子数:", real_mol_h.GetNumAtoms())

writer = Chem.SDWriter("osimertinib_6lud_crystal_pose.sdf")
writer.write(real_mol_h)
writer.close()
print("已写入 osimertinib_6lud_crystal_pose.sdf")

## 第5步：定义真实的蛋白-配体复合物MD函数(和core/md_stability.py::run_protein_ligand_complex_md()逻辑一致，加了显式GPU平台选择)

**真实踩过的坑1**：`GAFFTemplateGenerator(molecules=[off_mol])`不传`forcefield`参数时，在某些`openmmforcefields`版本上会报`TypeError: A GAFF force field name must be provided as a string`——不同版本对"不传forcefield该用哪个默认值"这件事处理得不一样(本地conda环境装的版本能自动补上`gaff-2.11`，Colab装到的版本不会)。下面已经改成显式传`forcefield="gaff-2.11"`，不依赖任何版本的默认行为。

**真实踩过的坑2**：`Platform.getNumPlatforms()`列出的平台里有`CUDA`，不代表真的能用——这次会话真实遇到过`OpenMMException: Error loading CUDA module: CUDA_ERROR_UNSUPPORTED_PTX_VERSION`，意思是conda装的openmm-cuda插件编译时用的CUDA版本比这台Colab机器实际的GPU驱动支持的版本更新，插件能加载、能列出来，但真正建Context执行计算时才崩。`get_best_platform()`已经改成不只是看列表选名字，是真的建一个最小的测试Context验证每个候选平台能不能用，第一个能用的才返回——CUDA真失败会自动退到OpenCL(真实在T4上也是GPU加速，只是走的是另一套驱动接口，不受这个PTX编译版本问题影响)，不会再直接崩在第6步。

**真实踩过的坑3(这个debug了5轮才定位到底)**：`from openff.toolkit import Molecule`会报`RuntimeError: operator torchvision::nms does not exist`。追完整堆栈(不是折叠版)才看清楚：openff.toolkit导入时会自动尝试注册`NAGLToolkitWrapper`(一个我们根本不用的ML电荷后端)，它的`is_available()`想`import openff.nagl`、本该用`try/except ImportError`兜底不可用的情况，但这条链路一路拖出`pytorch_lightning`→`torchmetrics`→`torchvision`，torchvision自己的原生扩展在这个环境里没能正确注册`torchvision::nms`这个op，崩的是`RuntimeError`不是`ImportError`，`is_available()`的兜底网完全接不住。前4次尝试(钉Python版本/卸载torch/补装torchvision/换`openff-toolkit-base`)都是在conda安装层面打转，后来发现`openmmforcefields`硬依赖完整版`openff-toolkit`，根本绕不开。**真正有效的修复**：纯Python运行时技巧，在`import openff.nagl`真正发生之前，把`sys.modules["openff.nagl"]`设成`None`——Python的import系统看到这个会直接对这次import抛出`ImportError`(这是文档化的标准行为，不是hack)，`is_available()`这次真的能接住，从根上避开整条broken链路，不碰conda环境、不删任何已装的包。下面已经加上这两行。

In [ ]:
import time
import numpy as np
from pdbfixer import PDBFixer
from openmm.app import ForceField, HBonds, Modeller, NoCutoff, Simulation, DCDReporter
from openmm import LangevinMiddleIntegrator, Platform
from openmm.unit import kelvin, nanometer, picosecond, picoseconds
from openmmforcefields.generators import GAFFTemplateGenerator

# 真实查到的坑：openff.toolkit导入时会自动尝试加载NAGLToolkitWrapper(一个我们
# 根本不用的ML电荷后端)，它的is_available()会import openff.nagl，这条链路会一路
# 拖出pytorch-lightning->torchmetrics->torchvision，torchvision自己的原生扩展
# 没能正确注册torchvision::nms这个op，导致崩溃(完整堆栈这次会话真实追踪到底过)。
# is_available()本来想用try/except ImportError兜底，但这里崩的是RuntimeError不是
# ImportError，兜不住。直接让"import openff.nagl"这行在到达之前就返回ImportError，
# 从根上避开整条broken链路，不碰conda环境、不删任何已装的包，纯Python运行时技巧：
import sys
sys.modules["openff.nagl"] = None

from openff.toolkit import Molecule


def get_best_platform():
    """依次尝试CUDA/OpenCL/CPU/Reference，真的建一个最小Context验证能不能用，
    不只是看platform在不在列表里——有的环境CUDA插件装上了、Platform.getNumPlatforms()
    会列出CUDA，但真正创建Context时才发现PTX版本和驱动不匹配(这次会话真实遇到的
    OpenMMException: CUDA_ERROR_UNSUPPORTED_PTX_VERSION就是这种情况)，这时候应该
    自动退到下一个真的能用的平台，不是直接崩掉。"""
    from openmm import System, VerletIntegrator, Context

    names = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
    for preferred in ("CUDA", "OpenCL", "CPU", "Reference"):
        if preferred not in names:
            continue
        try:
            test_system = System()
            test_system.addParticle(1.0)
            test_integrator = VerletIntegrator(0.001)
            test_platform = Platform.getPlatformByName(preferred)
            test_context = Context(test_system, test_integrator, test_platform)
            del test_context
            return test_platform
        except Exception as exc:
            print(f"平台{preferred}列在可用列表里，但真实创建Context失败({exc})，换下一个")
            continue
    return Platform.getPlatform(0)


def run_protein_ligand_complex_md(
    receptor_pdb_path, ligand_sdf_path, out_trajectory_path, n_steps=5000, report_interval=500
):
    """和core/md_stability.py::run_protein_ligand_complex_md()同一套逻辑，
    这里加了显式platform选择，确保真的用上GPU(如果有)。"""
    fixer = PDBFixer(filename=str(receptor_pdb_path))
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)

    lig_mol = Chem.MolFromMolFile(str(ligand_sdf_path), removeHs=False)
    off_mol = Molecule.from_rdkit(lig_mol, allow_undefined_stereo=True)
    off_mol.assign_partial_charges(partial_charge_method="gasteiger")
    gaff = GAFFTemplateGenerator(molecules=[off_mol], forcefield="gaff-2.11")  # 显式传forcefield，见下方真实踩过的坑

    modeller = Modeller(fixer.topology, fixer.positions)
    lig_topology = off_mol.to_topology().to_openmm()
    lig_positions = lig_mol.GetConformer().GetPositions() * 0.1 * nanometer
    modeller.add(lig_topology, lig_positions)

    forcefield = ForceField("amber14-all.xml", "implicit/gbn2.xml")
    forcefield.registerTemplateGenerator(gaff.generator)
    system = forcefield.createSystem(modeller.topology, nonbondedMethod=NoCutoff, constraints=HBonds)

    integrator = LangevinMiddleIntegrator(300 * kelvin, 1 / picosecond, 0.002 * picoseconds)
    platform = get_best_platform()
    print("真实使用的平台:", platform.getName())
    simulation = Simulation(modeller.topology, system, integrator, platform)
    simulation.context.setPositions(modeller.positions)

    t0 = time.time()
    energy_before = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    simulation.minimizeEnergy(maxIterations=300)
    energy_after = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    t_minimize = time.time() - t0

    standard_residues = {
        "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE", "LEU",
        "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL", "HOH",
    }
    ligand_atom_indices = [a.index for a in modeller.topology.atoms() if a.residue.name not in standard_residues]

    initial_positions = np.array(simulation.context.getState(getPositions=True).getPositions().value_in_unit(nanometer))
    simulation.context.setVelocitiesToTemperature(300 * kelvin)
    simulation.reporters.append(DCDReporter(str(out_trajectory_path), report_interval))

    rmsd_trace = []
    n_blocks = max(1, n_steps // report_interval)
    t1 = time.time()
    for _ in range(n_blocks):
        simulation.step(report_interval)
        pos = np.array(simulation.context.getState(getPositions=True).getPositions().value_in_unit(nanometer))
        diff = pos[ligand_atom_indices] - initial_positions[ligand_atom_indices]
        rmsd_angstrom = float(np.sqrt(np.mean(np.sum(diff ** 2, axis=1))) * 10)
        rmsd_trace.append(round(rmsd_angstrom, 2))
    t_dynamics = time.time() - t1

    return {
        "ok": True,
        "platform": platform.getName(),
        "n_total_atoms": modeller.topology.getNumAtoms(),
        "n_ligand_atoms": len(ligand_atom_indices),
        "n_steps_run": n_blocks * report_interval,
        "energy_before_minimization_kj_mol": energy_before.value_in_unit(energy_before.unit),
        "energy_after_minimization_kj_mol": energy_after.value_in_unit(energy_after.unit),
        "ligand_rmsd_trace_angstrom": rmsd_trace,
        "minimize_seconds": round(t_minimize, 1),
        "dynamics_seconds": round(t_dynamics, 1),
        "seconds_per_1000_steps": round(t_dynamics / (n_blocks * report_interval) * 1000, 2),
    }


print("函数定义完成")

## 第6步：先跑一个小规模标定(5000步=10ps)，真实测出这块GPU的吞吐量

和本地Mac CPU那次(86秒/10ps)做对比——这一步的意义是拿到真实数字，不是猜GPU能快多少。

In [ ]:
calib = run_protein_ligand_complex_md(
    receptor_pdb_path="6lud_receptor_only.pdb",
    ligand_sdf_path="osimertinib_6lud_crystal_pose.sdf",
    out_trajectory_path="calibration.dcd",
    n_steps=5000,
    report_interval=500,
)
import json
print(json.dumps(calib, indent=2))

cpu_seconds_per_10ps = 86.2  # 这次会话本地Mac CPU真实测出来的数字
gpu_seconds_per_10ps = calib["dynamics_seconds"]
speedup = cpu_seconds_per_10ps / gpu_seconds_per_10ps
print(f"\n真实加速比: {speedup:.1f}x (本地CPU 10ps={cpu_seconds_per_10ps}秒, 这块{calib['platform']} 10ps={gpu_seconds_per_10ps}秒)")

# 换算20ns(真实项目量级)大概需要多久
seconds_for_20ns = calib["seconds_per_1000_steps"] * (20000 / 0.002) / 1000
print(f"按这个速度推算，跑满20ns大概需要 {seconds_for_20ns/60:.1f} 分钟")

## 第7步：根据上一步真实测出来的速度，跑一个更接近真实项目量级的版本

默认设了一个20分钟的时间预算上限(Colab免费层会话有限，不要一次跑太久)，自动按标定结果换算成对应的步数——如果你想跑满20ns，把`time_budget_minutes`改大，但要注意免费层大概12小时会断，而且空闲太久也会断线。

In [ ]:
time_budget_minutes = 20  # 按需调整

n_steps_for_budget = int((time_budget_minutes * 60) / calib["seconds_per_1000_steps"] * 1000)
ns_covered = n_steps_for_budget * 0.002 / 1000
print(f"按{time_budget_minutes}分钟预算，这次会跑{n_steps_for_budget}步，覆盖约{ns_covered:.2f}ns的真实轨迹")

real_result = run_protein_ligand_complex_md(
    receptor_pdb_path="6lud_receptor_only.pdb",
    ligand_sdf_path="osimertinib_6lud_crystal_pose.sdf",
    out_trajectory_path="osimertinib_6lud_complex_md_gpu.dcd",
    n_steps=n_steps_for_budget,
    report_interval=max(500, n_steps_for_budget // 50),
)
print(json.dumps(real_result, indent=2))

## 第8步：把真实跑出来的轨迹文件下载回本地

`.dcd`文件可以用VMD/PyMOL/MDAnalysis这类工具打开查看真实的分子运动轨迹。

In [ ]:
from google.colab import files

files.download("osimertinib_6lud_complex_md_gpu.dcd")
print("真实配体RMSD轨迹:", real_result["ligand_rmsd_trace_angstrom"])
print("\n如果这条轨迹的RMSD数值一直在爬升、没有趋于平稳，说明这个姿态在更长时间尺度下可能不稳定——")
print("这正是环节4 L3(MD稳定性筛)想要回答的问题，现在是用真实GPU跑出来的真实数字，不是demo了。")

## 第9步：跑你自己真实筛选出来的候选分子(不是demo分子)

上面第1-8步跑的是奥希替尼+6LUD的demo，用来验证GPU工具链本身通不通。
这一节换成**你本仓库真实筛选出来的候选分子**：本地先用
`scripts/export_top_candidates_for_gpu_md.py`从真实数据库(`funnel_scores`
表里真实的Vina L1对接分数)导出打分最好的几个候选，连同它们真实的对接
姿态(不是重新生成的构象，是Vina真实算出来的姿态)和对应受体，打包成一个
zip——上传到这里继续跑。

**本地先跑这一步**(在你自己电脑的终端，不是在Colab里)：
```bash
PYTHONPATH=. .venv310/bin/python scripts/export_top_candidates_for_gpu_md.py \
    --route route_a_shp2_sos1 --top-n 3
# 会生成 validation/gpu_md_export/route_a_shp2_sos1_top3.zip
```
`--route`换成`route_c_4th_gen_tki`就是四代TKI那条路线的真实候选，
`--top-n`控制导出几个(按真实Vina打分从好到差排序取前N个)。生成的zip
在下面这个cell里上传。

In [ ]:
from google.colab import files
import zipfile, glob, os

print("选择刚才本地生成的 *_topN.zip 文件上传")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

extract_dir = "my_candidates"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(extract_dir)

print("\n解压出的文件:")
for f in sorted(os.listdir(extract_dir)):
    print(" -", f)

with open(os.path.join(extract_dir, "MANIFEST.txt")) as f:
    print("\n真实来源记录(MANIFEST.txt):\n" + f.read())


## 第10步：对每个真实候选分子跑复合物MD，用的是第5步定义的同一个函数

In [ ]:
import glob, os, json

if "calib" not in dir():
    raise NameError(
        "calib没有定义——这个数字来自第6步的标定cell(5000步demo跑一下测真实GPU速度)。\n"
        "如果你刚重启过运行时，第1-5步的变量都被清空了，需要在上传zip(第9步)之前，\n"
        "先依次重跑一遍第2/3/4/5/6步(不用跑第7/8步的完整demo+下载，第6步的标定\n"
        "结果就够用了)，再回来重跑第9步上传、第10步这里。"
    )

receptor_candidates = [f for f in os.listdir(extract_dir) if f.endswith(".pdb")]
assert len(receptor_candidates) == 1, f"受体文件应该只有1个，实际找到: {receptor_candidates}"
receptor_path = os.path.join(extract_dir, receptor_candidates[0])
print("真实受体:", receptor_path)

ligand_sdfs = sorted(glob.glob(os.path.join(extract_dir, "*.sdf")))
print(f"真实候选分子: {len(ligand_sdfs)}个 -> {[os.path.basename(p) for p in ligand_sdfs]}")

# 按第6步标定出来的真实GPU吞吐量换算步数，默认每个候选给10分钟预算
per_candidate_minutes = 10
n_steps_each = int((per_candidate_minutes * 60) / calib["seconds_per_1000_steps"] * 1000)
print(f"每个候选分配{per_candidate_minutes}分钟，约等于{n_steps_each}步"
      f"(≈{n_steps_each * 0.002 / 1000:.2f}ns)")

my_results = {}
for sdf_path in ligand_sdfs:
    compound_id = os.path.splitext(os.path.basename(sdf_path))[0]
    print(f"\n=== 真实跑 {compound_id} ===")
    result = run_protein_ligand_complex_md(
        receptor_pdb_path=receptor_path,
        ligand_sdf_path=sdf_path,
        out_trajectory_path=f"{compound_id}_md.dcd",
        n_steps=n_steps_each,
        report_interval=max(1, n_steps_each // 20),
    )
    my_results[compound_id] = result
    if result.get("ok", True) and "ligand_rmsd_trace_angstrom" in result:
        trace = result["ligand_rmsd_trace_angstrom"]
        mean_rmsd = sum(trace) / len(trace)
        verdict = "稳定(mean<2.5Å)" if mean_rmsd < 2.5 else "可能不稳定(mean>=2.5Å)"
        print(f"{compound_id}: 配体RMSD轨迹={trace}, 均值={mean_rmsd:.2f}Å -> {verdict}")
    else:
        print(f"{compound_id}: 失败 - {result}")

with open("my_candidates_md_results.json", "w") as f:
    json.dump(my_results, f, indent=2)

## 第11步：把所有真实候选的轨迹+结果汇总下载回本地

对照本仓库`doc/04-funnel-docking.md`的L3 pass criteria(配体RMSD均值<2.5Å)
自己判断哪些候选在这个时间尺度上是稳定的——这是真实的筛选结果，不是demo，
可以直接拿去更新`funnel_scores`表里对应候选的L3记录。

In [ ]:
import zipfile

out_zip = "my_candidates_gpu_md_results.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("my_candidates_md_results.json")
    for sdf_path in ligand_sdfs:
        compound_id = os.path.splitext(os.path.basename(sdf_path))[0]
        dcd_path = f"{compound_id}_md.dcd"
        if os.path.exists(dcd_path):
            zf.write(dcd_path)

files.download(out_zip)
print(f"真实结果已打包: {out_zip}，下载后可以把my_candidates_md_results.json")
print("里每个候选的ligand_rmsd_trace_angstrom贴回来，我可以帮你解读、更新到文档/数据库里。")
